In [1]:
# =========================================
# IMPORT LIBRARIES
# =========================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import boto3
from botocore.client import Config

In [2]:
# =========================================
# INIT SPARK SESSION
# =========================================

spark = SparkSession.builder \
    .appName("SV3_Crypto_ETL") \
    .getOrCreate()

print("Spark Started Successfully")

# =========================================
# MINIO S3A CONFIG
# =========================================

hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.access.key", "admin")
hadoop_conf.set("fs.s3a.secret.key", "password123")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")

hadoop_conf.set(
    "fs.s3a.aws.credentials.provider",
    "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
)

print("MinIO Configuration Completed")

Spark Started Successfully
MinIO Configuration Completed


In [3]:
# =========================================
# LOAD RAW DATA FROM MINIO
# =========================================

df = spark.read.csv(
    "s3a://crypto-raw-data/bitcoin_1m.csv",
    header=True,
    inferSchema=True
)

print("Raw Dataset Loaded")

print("Total Rows:", df.count())

df.show(20)

df.printSchema()

Raw Dataset Loaded
Total Rows: 1000
+-------------------+--------+--------+--------+--------+----------+
|          timestamp|    open|    high|     low|   close|    volume|
+-------------------+--------+--------+--------+--------+----------+
|2026-06-10 17:40:00|61246.73| 61251.6|61213.85|61215.28|0.09241264|
|2026-06-10 17:41:00|61215.41|61270.39|61215.41|61257.45|0.92260214|
|2026-06-10 17:42:00|61251.16| 61282.0|61251.16| 61282.0|0.10979783|
|2026-06-10 17:43:00| 61282.0|61289.93|61270.36|61270.36|0.17064752|
|2026-06-10 17:44:00| 61258.4|61263.87|61253.75|61263.87|0.02606856|
|2026-06-10 17:45:00|61264.67|61283.22|61263.46|61266.03|0.00258343|
|2026-06-10 17:46:00|61278.66| 61309.6|61269.43|61270.38|0.55335698|
|2026-06-10 17:47:00| 61271.7|61297.12|61271.48|61271.48|0.13833864|
|2026-06-10 17:48:00|61271.45|61276.13|61239.78|61256.73|0.19317689|
|2026-06-10 17:49:00|61256.45|61289.73|61256.44|61288.78|0.21268462|
|2026-06-10 17:50:00|61289.15|61312.99| 61288.8|61300.85|0.35180617

In [4]:
# =========================================
# NULL CHECK
# =========================================

print("NULL CHECK")

df.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
]).show()

# =========================================
# DUPLICATE CHECK
# =========================================

total_rows = df.count()

unique_rows = df.dropDuplicates(
    ["timestamp"]
).count()

print("Total Rows:", total_rows)
print("Unique Rows:", unique_rows)
print("Duplicates:", total_rows - unique_rows)

NULL CHECK
+---------+----+----+---+-----+------+
|timestamp|open|high|low|close|volume|
+---------+----+----+---+-----+------+
|        0|   0|   0|  0|    0|     0|
+---------+----+----+---+-----+------+

Total Rows: 1000
Unique Rows: 1000
Duplicates: 0


In [5]:
# =========================================
# STANDARDIZE TIMESTAMP
# =========================================

df = df.withColumn("timestamp", F.to_timestamp("timestamp"))
df = df.orderBy("timestamp")

df.select(
    F.min("timestamp").alias("min_time"),
    F.max("timestamp").alias("max_time")
).show(truncate=False)

+-------------------+-------------------+
|min_time           |max_time           |
+-------------------+-------------------+
|2026-06-10 17:40:00|2026-06-11 10:19:00|
+-------------------+-------------------+



In [6]:
# =========================================
# GAP DETECTION
# =========================================

w = Window.orderBy("timestamp")

df = df.withColumn("prev_time", F.lag("timestamp").over(w))

df = df.withColumn(
    "diff_min",
    (F.unix_timestamp("timestamp")
     - F.unix_timestamp("prev_time")) / 60
)

df.select(
    "timestamp",
    "prev_time",
    "diff_min"
).show(20, False)

gap_count = df.filter(F.col("diff_min") > 1).count()

print("Gap Count:", gap_count)

+-------------------+-------------------+--------+
|timestamp          |prev_time          |diff_min|
+-------------------+-------------------+--------+
|2026-06-10 17:40:00|NULL               |NULL    |
|2026-06-10 17:41:00|2026-06-10 17:40:00|1.0     |
|2026-06-10 17:42:00|2026-06-10 17:41:00|1.0     |
|2026-06-10 17:43:00|2026-06-10 17:42:00|1.0     |
|2026-06-10 17:44:00|2026-06-10 17:43:00|1.0     |
|2026-06-10 17:45:00|2026-06-10 17:44:00|1.0     |
|2026-06-10 17:46:00|2026-06-10 17:45:00|1.0     |
|2026-06-10 17:47:00|2026-06-10 17:46:00|1.0     |
|2026-06-10 17:48:00|2026-06-10 17:47:00|1.0     |
|2026-06-10 17:49:00|2026-06-10 17:48:00|1.0     |
|2026-06-10 17:50:00|2026-06-10 17:49:00|1.0     |
|2026-06-10 17:51:00|2026-06-10 17:50:00|1.0     |
|2026-06-10 17:52:00|2026-06-10 17:51:00|1.0     |
|2026-06-10 17:53:00|2026-06-10 17:52:00|1.0     |
|2026-06-10 17:54:00|2026-06-10 17:53:00|1.0     |
|2026-06-10 17:55:00|2026-06-10 17:54:00|1.0     |
|2026-06-10 17:56:00|2026-06-10

In [7]:
# =========================================
# MA10 & MA60
# =========================================

w10 = Window.orderBy("timestamp").rowsBetween(-9, 0)
w60 = Window.orderBy("timestamp").rowsBetween(-59, 0)

df = df.withColumn("MA10", F.avg("close").over(w10))
df = df.withColumn("MA60", F.avg("close").over(w60))

df.select(
    "timestamp",
    "close",
    "MA10",
    "MA60"
).show(20, False)

+-------------------+--------+------------------+------------------+
|timestamp          |close   |MA10              |MA60              |
+-------------------+--------+------------------+------------------+
|2026-06-10 17:40:00|61215.28|61215.28          |61215.28          |
|2026-06-10 17:41:00|61257.45|61236.365         |61236.365         |
|2026-06-10 17:42:00|61282.0 |61251.57666666666 |61251.57666666666 |
|2026-06-10 17:43:00|61270.36|61256.27249999999 |61256.27249999999 |
|2026-06-10 17:44:00|61263.87|61257.791999999994|61257.791999999994|
|2026-06-10 17:45:00|61266.03|61259.165         |61259.165         |
|2026-06-10 17:46:00|61270.38|61260.76714285714 |61260.76714285714 |
|2026-06-10 17:47:00|61271.48|61262.10625       |61262.10625       |
|2026-06-10 17:48:00|61256.73|61261.508888888886|61261.508888888886|
|2026-06-10 17:49:00|61288.78|61264.236         |61264.236         |
|2026-06-10 17:50:00|61300.85|61272.79299999999 |61267.564545454545|
|2026-06-10 17:51:00|61320.74|6127

In [8]:
# =========================================
# ROC + MOMENTUM
# =========================================

w = Window.orderBy("timestamp")

df = df.withColumn("close_lag10", F.lag("close", 10).over(w))

df = df.withColumn(
    "ROC",
    (F.col("close") - F.col("close_lag10"))
    / F.col("close_lag10") * 100
)

df = df.withColumn(
    "MOM",
    F.col("close") - F.col("close_lag10")
)

df.select(
    "close",
    "close_lag10",
    "ROC",
    "MOM"
).show(20, False)

+--------+-----------+---------------------+-------------------+
|close   |close_lag10|ROC                  |MOM                |
+--------+-----------+---------------------+-------------------+
|61215.28|NULL       |NULL                 |NULL               |
|61257.45|NULL       |NULL                 |NULL               |
|61282.0 |NULL       |NULL                 |NULL               |
|61270.36|NULL       |NULL                 |NULL               |
|61263.87|NULL       |NULL                 |NULL               |
|61266.03|NULL       |NULL                 |NULL               |
|61270.38|NULL       |NULL                 |NULL               |
|61271.48|NULL       |NULL                 |NULL               |
|61256.73|NULL       |NULL                 |NULL               |
|61288.78|NULL       |NULL                 |NULL               |
|61300.85|61215.28   |0.13978536077920367  |85.56999999999971  |
|61320.74|61257.45   |0.10331804539692865  |63.29000000000087  |
|61292.03|61282.0    |0.0

In [9]:
# =========================================
# RSI 14
# =========================================

w1 = Window.orderBy("timestamp")
w14 = Window.orderBy("timestamp").rowsBetween(-13, 0)

df = df.withColumn("change", F.col("close") - F.lag("close").over(w1))

df = df.withColumn(
    "gain",
    F.when(F.col("change") > 0, F.col("change")).otherwise(0)
)

df = df.withColumn(
    "loss",
    F.when(F.col("change") < 0, -F.col("change")).otherwise(0)
)

df = df.withColumn("avg_gain", F.avg("gain").over(w14))
df = df.withColumn("avg_loss", F.avg("loss").over(w14))

df = df.withColumn(
    "RS",
    F.when(F.col("avg_loss") == 0, None)
     .otherwise(F.col("avg_gain") / F.col("avg_loss"))
)

df = df.withColumn(
    "RSI",
    F.when(F.col("avg_loss") == 0, 100)
     .when(F.col("avg_gain") == 0, 0)
     .otherwise(
         100 - (100 / (1 + F.col("RS")))
     )
)

print("RSI Created")

df.select(
    "timestamp",
    "close",
    "change",
    "gain",
    "loss",
    "avg_gain",
    "avg_loss",
    "RS",
    "RSI"
).show(20, False)

RSI Created
+-------------------+--------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+
|timestamp          |close   |change             |gain              |loss              |avg_gain          |avg_loss          |RS                |RSI               |
+-------------------+--------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+
|2026-06-10 17:40:00|61215.28|NULL               |0.0               |0.0               |0.0               |0.0               |NULL              |100.0             |
|2026-06-10 17:41:00|61257.45|42.169999999998254 |42.169999999998254|0.0               |21.084999999999127|0.0               |NULL              |100.0             |
|2026-06-10 17:42:00|61282.0 |24.55000000000291  |24.55000000000291 |0.0               |22.24000000000039 |0.0               |NULL              |100.0             

In [10]:
# =========================================
# STOCHASTIC OSCILLATOR
# =========================================

df = df.withColumn(
    "highest_high",
    F.max("high").over(w14)
)

df = df.withColumn(
    "lowest_low",
    F.min("low").over(w14)
)

df = df.withColumn(
    "stoch_k",
    F.when(
        (F.col("highest_high") - F.col("lowest_low")) == 0,
        None
    ).otherwise(
        (F.col("close") - F.col("lowest_low"))
        /
        (F.col("highest_high") - F.col("lowest_low"))
        * 100
    )
)

w3 = Window.orderBy("timestamp").rowsBetween(-2, 0)

df = df.withColumn(
    "stoch_d",
    F.avg("stoch_k").over(w3)
)

print("Stochastic Created")

df.select(
    "timestamp",
    "close",
    "highest_high",
    "lowest_low",
    "stoch_k",
    "stoch_d"
).show(20, False)

Stochastic Created
+-------------------+--------+------------+----------+------------------+------------------+
|timestamp          |close   |highest_high|lowest_low|stoch_k           |stoch_d           |
+-------------------+--------+------------+----------+------------------+------------------+
|2026-06-10 17:40:00|61215.28|61251.6     |61213.85  |3.7880794701994467|3.7880794701994467|
|2026-06-10 17:41:00|61257.45|61270.39    |61213.85  |77.1135479306648  |40.45081370043212 |
|2026-06-10 17:42:00|61282.0 |61282.0     |61213.85  |100.0             |60.300542466954745|
|2026-06-10 17:43:00|61270.36|61289.93    |61213.85  |74.27707676130485 |83.79687489732322 |
|2026-06-10 17:44:00|61263.87|61289.93    |61213.85  |65.74658254469364 |80.00788643533282 |
|2026-06-10 17:45:00|61266.03|61289.93    |61213.85  |68.58569926393152 |69.53645285664334 |
|2026-06-10 17:46:00|61270.38|61309.6     |61213.85  |59.03916449086041 |64.4571487664952  |
|2026-06-10 17:47:00|61271.48|61309.6     |61213.85

In [11]:
# =========================================
# BUY / SELL LABEL
# =========================================

df = df.withColumn(
    "label",
    F.when(F.col("MA10") > F.col("MA60"), 1).otherwise(0)
)

print("Buy/Sell Label Created")

print("Label Distribution")

df.groupBy("label").count().show()

df.select(
    "timestamp",
    "MA10",
    "MA60",
    "label"
).show(20, False)

Buy/Sell Label Created
Label Distribution
+-----+-----+
|label|count|
+-----+-----+
|    0|  519|
|    1|  481|
+-----+-----+

+-------------------+------------------+------------------+-----+
|timestamp          |MA10              |MA60              |label|
+-------------------+------------------+------------------+-----+
|2026-06-10 17:40:00|61215.28          |61215.28          |0    |
|2026-06-10 17:41:00|61236.365         |61236.365         |0    |
|2026-06-10 17:42:00|61251.57666666666 |61251.57666666666 |0    |
|2026-06-10 17:43:00|61256.27249999999 |61256.27249999999 |0    |
|2026-06-10 17:44:00|61257.791999999994|61257.791999999994|0    |
|2026-06-10 17:45:00|61259.165         |61259.165         |0    |
|2026-06-10 17:46:00|61260.76714285714 |61260.76714285714 |0    |
|2026-06-10 17:47:00|61262.10625       |61262.10625       |0    |
|2026-06-10 17:48:00|61261.508888888886|61261.508888888886|0    |
|2026-06-10 17:49:00|61264.236         |61264.236         |0    |
|2026-06-10 17:

In [12]:
# =========================================
# FEATURE TABLE
# =========================================

final_df = df.select(
    "timestamp",
    "open", "high", "low", "close", "volume",
    "MA10", "MA60",
    "ROC", "MOM",
    "RSI",
    "stoch_k", "stoch_d",
    "label"
)

print("Rows Before DropNA:", final_df.count())

final_df = final_df.dropna()

print("Rows After DropNA:", final_df.count())

Rows Before DropNA: 1000
Rows After DropNA: 990


In [13]:
# =========================================
# CREATE MINIO BUCKET
# =========================================

s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="admin",
    aws_secret_access_key="password123",
    config=Config(signature_version="s3v4")
)

bucket_name = "crypto-feature-table"

if bucket_name not in [
    b["Name"]
    for b in s3.list_buckets()["Buckets"]
]:
    s3.create_bucket(
        Bucket=bucket_name
    )
    print("Bucket Created")
else:
    print("Bucket Already Exists")

Bucket Already Exists


In [14]:
# =========================================
# SAVE FEATURE TABLE
# =========================================

final_df.write \
    .mode("overwrite") \
    .parquet(
        "s3a://crypto-feature-table/features/"
    )

print("Feature Table Saved")

Feature Table Saved


In [15]:
# =========================================
# VERIFY OUTPUT
# =========================================

verify_df = spark.read.parquet(
    "s3a://crypto-feature-table/features/"
)

print(
    "Rows Written:",
    verify_df.count()
)

verify_df.show(30, False)

verify_df.printSchema()

Rows Written: 990
+-------------------+--------+--------+--------+--------+-----------+------------------+------------------+---------------------+-------------------+------------------+------------------+------------------+-----+
|timestamp          |open    |high    |low     |close   |volume     |MA10              |MA60              |ROC                  |MOM                |RSI               |stoch_k           |stoch_d           |label|
+-------------------+--------+--------+--------+--------+-----------+------------------+------------------+---------------------+-------------------+------------------+------------------+------------------+-----+
|2026-06-10 17:50:00|61289.15|61312.99|61288.8 |61300.85|0.35180617 |61272.79299999999 |61267.564545454545|0.13978536077920367  |85.56999999999971  |78.27264917729559 |87.75469033689784 |70.26461827592153 |1    |
|2026-06-10 17:51:00|61317.1 |61322.98|61305.92|61320.74|0.3802602  |61279.121999999996|61271.99583333333 |0.10331804539692865  |6

In [20]:
df = spark.read.parquet(
    "s3a://crypto-feature-table/features/"
)

In [21]:
df.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: double (nullable = true)
 |-- MA10: double (nullable = true)
 |-- MA60: double (nullable = true)
 |-- ROC: double (nullable = true)
 |-- MOM: double (nullable = true)
 |-- RSI: double (nullable = true)
 |-- stoch_k: double (nullable = true)
 |-- stoch_d: double (nullable = true)
 |-- label: integer (nullable = true)



In [22]:
df.show(20, truncate=False)

+-------------------+--------+--------+--------+--------+-----------+------------------+------------------+---------------------+-------------------+------------------+------------------+------------------+-----+
|timestamp          |open    |high    |low     |close   |volume     |MA10              |MA60              |ROC                  |MOM                |RSI               |stoch_k           |stoch_d           |label|
+-------------------+--------+--------+--------+--------+-----------+------------------+------------------+---------------------+-------------------+------------------+------------------+------------------+-----+
|2026-06-10 17:50:00|61289.15|61312.99|61288.8 |61300.85|0.35180617 |61272.79299999999 |61267.564545454545|0.13978536077920367  |85.56999999999971  |78.27264917729559 |87.75469033689784 |70.26461827592153 |1    |
|2026-06-10 17:51:00|61317.1 |61322.98|61305.92|61320.74|0.3802602  |61279.121999999996|61271.99583333333 |0.10331804539692865  |63.29000000000087  